# Homework 6 — Decision Trees and Ensemble Learning

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

def rmse(y, y_pred):
    return np.sqrt(mean_squared_error(y, y_pred))

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)
target = 'fuel_efficiency_mpg'
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [2]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [3]:
df = df.fillna(0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
y_train = df_train[target].values
y_val = df_val[target].values

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.drop(columns=target).to_dict(orient='records'))
X_val = dv.transform(df_val.drop(columns=target).to_dict(orient='records'))
features = list(dv.get_feature_names_out())
X_train.shape, X_val.shape

((6000, 16), (2000, 16))

## Q1. Decision tree with `max_depth=1`

In [4]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

split_feature = features[dt.tree_.feature[0]]
print(export_text(dt, feature_names=features))
print('original column:', split_feature.split('=')[0])

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]

original column: model_year


## Q2. Random forest, `n_estimators=10`

In [5]:
def rmse_rf(n_estimators, max_depth=None):
    rf = RandomForestRegressor(
        n_estimators=n_estimators, max_depth=max_depth, random_state=1, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    return rmse(y_val, rf.predict(X_val))

rmse_rf(10)

1.8370540547300178

## Q3. Number of estimators

In [6]:
for n in [10, 50, 100, 150]:
    print(n, round(rmse_rf(n), 3))

10 1.837


50 1.772


100 1.768


150 1.769


## Q4. Best `max_depth` by mean RMSE

In [7]:
for d in [10, 15, 20, 25]:
    scores = [rmse_rf(n, max_depth=d) for n in [10, 50, 100, 150]]
    print(d, round(np.mean(scores), 4))

10 1.7554


15 1.784


20 1.7882


25 1.7863


## Q5. Feature importance

In [8]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=features)
importances[['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']].sort_values(ascending=False)

vehicle_weight         0.203879
horsepower             0.078017
engine_displacement    0.061005
acceleration           0.055201
dtype: float64

## Q6. XGBoost: `eta` 0.3 vs 0.1

In [9]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

def train_xgb(eta):
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,

        'objective': 'reg:squarederror',
        'nthread': 8,

        'seed': 1,
        'verbosity': 1,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=False)
    return rmse(y_val, model.predict(dval))

for eta in [0.3, 0.1]:
    print(eta, round(train_xgb(eta), 4))

0.3 1.8265


0.1 1.7248
